In [2]:
!pip install ipywidgets 

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
%pip install ipywidgets 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- Data Generation ---
np.random.seed(42)
n = 240
departments = ["Computer Science", "IT", "Mechanical", "Civil", "Electrical"]
statuses = ["Active", "Inactive"]
first_names = [
    "Aarav", "Vivaan", "Aditya", "Arjun", "Rahul",
    "Rohan", "Priya", "Ananya", "Sneha", "Neha",
    "Amit", "Karan", "Pooja", "Simran", "Riya"
]
last_names = [
    "Sharma", "Verma", "Singh", "Kumar", "Gupta",
    "Yadav", "Patel", "Mishra", "Jain", "Das"
]

students = pd.DataFrame({
    "Roll No": [f"STU{1001+i}" for i in range(n)],
    "Name": [
        f"{np.random.choice(first_names)} {np.random.choice(last_names)}"
        for _ in range(n)
    ],
    "Department": np.random.choice(departments, n),
    "Year": np.random.randint(1, 5, n),
    "Attendance": np.random.randint(50, 101, n),
    "Marks": np.random.randint(30, 101, n),
    "Fees": np.random.choice([15000, 20000, 25000, 30000, 35000], n),
    "Fee Status": np.random.choice(["Paid", "Pending"], n, p=[0.7, 0.3]),
    "Status": np.random.choice(statuses, n, p=[0.85, 0.15])
})
students.to_csv("students_data.csv", index=False)
# --- UI Widgets ---
display(HTML("""
<h1 style="color:#2563eb;">Student Management Dashboard</h1>
<p>Manage students, attendance, results and fees in one place.</p>
"""))

section = widgets.ToggleButtons(
    options=["Overview", "Students", "Attendance", "Results", "Fees"],
    description="Section:"
)
search_box = widgets.Text(
    placeholder="Search by name or roll number",
    description="Search:"
)
department_filter = widgets.Dropdown(
    options=["All"] + departments,
    value="All",
    description="Department:"
)
status_filter = widgets.Dropdown(
    options=["All", "Active", "Inactive"],
    value="All",
    description="Status:"
)
output = widgets.Output()

display(section)
display(search_box)
display(department_filter)
display(status_filter)
display(output)

# --- Dashboard Display Functions ---
def show_kpis(data):
    total_students = len(data)
    average_attendance = data["Attendance"].mean() if total_students else 0
    average_marks = data["Marks"].mean() if total_students else 0
    pending_fees = data.loc[data["Fee Status"] == "Pending", "Fees"].sum()
    display(HTML(f"""
    <div style="display:flex;gap:12px;flex-wrap:wrap;">
        <div style="background:#dbeafe;padding:18px;border-radius:10px;min-width:140px;">
            <b>Total Students</b><h2>{total_students}</h2>
        </div>
        <div style="background:#dcfce7;padding:18px;border-radius:10px;min-width:140px;">
            <b>Avg Attendance</b><h2>{average_attendance:.1f}%</h2>
        </div>
        <div style="background:#fef3c7;padding:18px;border-radius:10px;min-width:140px;">
            <b>Average Marks</b><h2>{average_marks:.1f}</h2>
        </div>
        <div style="background:#fee2e2;padding:18px;border-radius:10px;min-width:140px;">
            <b>Pending Fees</b><h2>₹{pending_fees:,}</h2>
        </div>
    </div>
    """))

def show_overview(data):
    show_kpis(data)
    if data.empty:
        print("No student data available.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    data["Department"].value_counts().plot(
        kind="bar", ax=axes[0], title="Students by Department"
    )
    axes[0].set_xlabel("Department")
    axes[0].set_ylabel("Number of Students")
    axes[0].tick_params(axis="x", rotation=30)
    data["Fee Status"].value_counts().plot(
        kind="pie",
        autopct="%1.1f%%",
        ax=axes[1],
        title="Fee Status"
    )
    axes[1].set_ylabel("")
    plt.tight_layout()
    plt.show()
    display(data.head(10))

def show_students(data):
    display(HTML("<h3>Student Records</h3>"))
    if data.empty:
        print("No students found.")
    else:
        display(data)

def show_attendance(data):
    display(HTML("<h3>Attendance Report</h3>"))
    if data.empty:
        print("No attendance records found.")
        return
    attendance_data = data[
        ["Roll No", "Name", "Department", "Attendance"]
    ].copy()
    attendance_data["Attendance Status"] = np.where(
        attendance_data["Attendance"] >= 75,
        "Good",
        "Low"
    )
    display(attendance_data)
    attendance_data["Attendance"].plot(
        kind="hist",
        bins=10,
        figsize=(8, 4),
        title="Attendance Distribution"
    )
    plt.xlabel("Attendance Percentage")
    plt.show()

def show_results(data):
    display(HTML("<h3>Student Results</h3>"))
    if data.empty:
        print("No result records found.")
        return
    results = data[["Roll No", "Name", "Department", "Marks"]].copy()
    results["Grade"] = np.select(
        [
            results["Marks"] >= 90,
            results["Marks"] >= 75,
            results["Marks"] >= 60,
            results["Marks"] >= 40
        ],
        ["A+", "A", "B", "C"],
        default="F"
    )
    display(results)
    results["Marks"].plot(
        kind="hist",
        bins=10,
        figsize=(8, 4),
        title="Marks Distribution"
    )
    plt.xlabel("Marks")
    plt.show()

def show_fees(data):
    display(HTML("<h3>Student Fee Records</h3>"))
    if data.empty:
        print("No fee records found.")
        return
    fee_data = data[
        ["Roll No", "Name", "Department", "Fees", "Fee Status"]
    ]
    display(fee_data)
    pending = data.loc[
        data["Fee Status"] == "Pending", "Fees"
    ].sum()
    paid = data.loc[
        data["Fee Status"] == "Paid", "Fees"
    ].sum()
    print(f"Total Paid Fees: ₹{paid:,}")
    print(f"Total Pending Fees: ₹{pending:,}")

# --- Logic for Filtering and Updating ---
def get_filtered_data():
    data = students.copy()
    query = search_box.value.strip().lower()
    if query:
        data = data[
            data["Name"].str.lower().str.contains(query, na=False)
            | data["Roll No"].str.lower().str.contains(query, na=False)
        ]
    if department_filter.value != "All":
        data = data[
            data["Department"] == department_filter.value
        ]
    if status_filter.value != "All":
        data = data[
            data["Status"] == status_filter.value
        ]
    return data

def update_dashboard(change=None):
    with output:
        clear_output(wait=True)
        data = get_filtered_data()
        if section.value == "Overview":
            show_overview(data)
        elif section.value == "Students":
            show_students(data)
        elif section.value == "Attendance":
            show_attendance(data)
        elif section.value == "Results":
            show_results(data)
        elif section.value == "Fees":
            show_fees(data)

# --- Add Student Form Widgets ---
name_input = widgets.Text(description="Name:", placeholder="Enter student name")
roll_input = widgets.Text(description="Roll No:", placeholder="Enter unique roll number")
dept_input = widgets.Dropdown(options=departments, description="Department:")
year_input = widgets.Dropdown(options=[1, 2, 3, 4], description="Year:")
attendance_input = widgets.IntSlider(value=80, min=0, max=100, description="Attendance:")
marks_input = widgets.IntSlider(value=70, min=0, max=100, description="Marks:")
fees_input = widgets.IntText(value=20000, description="Fees:")
fee_status_input = widgets.Dropdown(options=["Paid", "Pending"], description="Fee Status:")
add_button = widgets.Button(description="Add Student", button_style="success")
add_output = widgets.Output()

display(
    name_input,
    roll_input,
    dept_input,
    year_input,
    attendance_input,
    marks_input,
    fees_input,
    fee_status_input,
    add_button,
    add_output
)

# --- Event Handlers ---
def add_student(button):
    global students
    with add_output:
        clear_output()
        name = name_input.value.strip()
        roll = roll_input.value.strip()
        if not name or not roll:
            print("Please enter both name and roll number.")
            return
        if students["Roll No"].str.lower().eq(roll.lower()).any():
            print("This roll number already exists.")
            return
        if fees_input.value < 0:
            print("Fees cannot be negative.")
            return
        new_student = pd.DataFrame([{
            "Roll No": roll,
            "Name": name,
            "Department": dept_input.value,
            "Year": year_input.value,
            "Attendance": attendance_input.value,
            "Marks": marks_input.value,
            "Fees": fees_input.value,
            "Fee Status": fee_status_input.value,
            "Status": "Active"
        }])
        students = pd.concat([students, new_student], ignore_index=True)
        print(f"Student {name} added successfully.")
        name_input.value = ""
        roll_input.value = ""
        update_dashboard()

section.observe(update_dashboard, names="value")
search_box.observe(update_dashboard, names="value")
department_filter.observe(update_dashboard, names="value")
status_filter.observe(update_dashboard, names="value")
add_button.on_click(add_student)

# Initial load
update_dashboard()

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


ToggleButtons(description='Section:', options=('Overview', 'Students', 'Attendance', 'Results', 'Fees'), value…

Text(value='', description='Search:', placeholder='Search by name or roll number')

Dropdown(description='Department:', options=('All', 'Computer Science', 'IT', 'Mechanical', 'Civil', 'Electric…

...

IntSlider(value=70, description='Marks:')

IntText(value=20000, description='Fees:')

Dropdown(description='Fee Status:', options=('Paid', 'Pending'), value='Paid')

Button(button_style='success', description='Add Student', style=ButtonStyle())

Output()